In [88]:
import torch
import torch.nn as nn

#### Masking

In [89]:
# masking a  normal matrix
mat1=torch.ones((5,5))
mat1

tensor([[1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1.]])

In [90]:
mask=torch.triu(mat1,diagonal=1)
masked=mat1.masked_fill(mask.bool(),-torch.inf) ## replace all 1 in mask with -inf

In [91]:
masked

tensor([[1., -inf, -inf, -inf, -inf],
        [1., 1., -inf, -inf, -inf],
        [1., 1., 1., -inf, -inf],
        [1., 1., 1., 1., -inf],
        [1., 1., 1., 1., 1.]])

In [92]:
torch.softmax(masked,dim=-1)

tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.5000, 0.5000, 0.0000, 0.0000, 0.0000],
        [0.3333, 0.3333, 0.3333, 0.0000, 0.0000],
        [0.2500, 0.2500, 0.2500, 0.2500, 0.0000],
        [0.2000, 0.2000, 0.2000, 0.2000, 0.2000]])

#### Dropout

In GPT/ Vanilla Transformer dropout is applied in 2 placed one is in attention matrix and other and contextual embedding

Dropout just makes sure some tokens are not fully dependent on the other one

After dropout some wts get removed while other get scaled

Eg: if p=1/6 1/6 of wts get removed and remaining other elements are scaled as A/ 1-p ie scaled by 6/5

In [93]:
mat1=torch.ones((5,5))
dropout=nn.Dropout(1/6)
mat1=dropout(mat1)
mat1

tensor([[1.2000, 1.2000, 1.2000, 0.0000, 1.2000],
        [1.2000, 1.2000, 1.2000, 1.2000, 1.2000],
        [1.2000, 1.2000, 1.2000, 1.2000, 1.2000],
        [1.2000, 0.0000, 1.2000, 1.2000, 0.0000],
        [1.2000, 0.0000, 1.2000, 0.0000, 0.0000]])

In [94]:
class Masked_attention(nn.Module):

    def __init__(self, d_in, d_out, qkv_bias=False):
        super().__init__()
        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key   = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.dim_key=d_out 
        self.dropout=nn.Dropout(0.5)

    def forward(self,x):
        
        key = self.W_key(x)
        query = self.W_query(x)
        value = self.W_value(x)
        attention_score= query @ key.transpose(-2, -1)
        ## masking
        seq_len=attention_score.shape[0]
        mask=torch.triu(torch.ones(seq_len,seq_len),diagonal=1)
        attention_score=attention_score.masked_fill(mask.bool(),-torch.inf)

        scaled_attention_score= torch.softmax(attention_score/torch.sqrt(torch.tensor(self.dim_key)),dim=-1)
        scaled_attention_score=self.dropout(scaled_attention_score)
        context_vector=scaled_attention_score @ value
        return context_vector


#### Masked self attention for a batches of input

In [95]:
class Masked_attention_v2(nn.Module):
    def __init__(self,dim_in,dim_out,dropout,context_length,qkv_bias=False):
        super().__init__()
        self.W_query = nn.Linear(dim_in, dim_out, bias=qkv_bias)
        self.W_key   = nn.Linear(dim_in, dim_out, bias=qkv_bias)
        self.W_value = nn.Linear(dim_in, dim_out, bias=qkv_bias)
        self.dim_key=dim_out 
        self.dropout=nn.Dropout(dropout)
        self.register_buffer('mask', torch.triu(torch.ones(context_length, context_length), diagonal=1)) ## mask is variable name
        ## upper trangular "mask" with all 1
    def forward(self,x):
        b,num_tokens,d_in=x.shape 
        ### so input will be like 1st batch then matrix then 2nd batch then matrix
        key = self.W_key(x)
        query = self.W_query(x)
        value = self.W_value(x)
        attention_score = query @ key.transpose(1, 2) # in 3d just swap dim1 and dim2(new transpose way)
        attention_score=attention_score.masked_fill(self.mask.bool()[:num_tokens,:num_tokens],-torch.inf) 
        ## in in complete sentences num_tokens>context_length so we mask acc to num_tokens in that batch
        
        scaled_attention_score= torch.softmax(attention_score/torch.sqrt(torch.tensor(self.dim_key)),dim=-1)
        scaled_attention_score=self.dropout(scaled_attention_score)
        context_vector=scaled_attention_score @ value
        return context_vector
           

In [96]:
inputs = torch.tensor(
  [[0.43, 0.15, 0.89], 
   [0.55, 0.87, 0.66], 
   [0.57, 0.85, 0.64], 
   [0.22, 0.58, 0.33], 
   [0.77, 0.25, 0.10], 
   [0.05, 0.80, 0.55]] 
)  
batch = torch.stack((inputs, inputs), dim=0)
print(batch.shape) 

torch.Size([2, 6, 3])


In [97]:
torch.manual_seed(123)
d_in=3
d_out=7
context_length = batch.shape[1]
masked_a = Masked_attention_v2(d_in, d_out,0.0,context_length)
context_vecs = masked_a(batch) ## forward pass 
print("Contextual vector shape:", context_vecs.shape)

Contextual vector shape: torch.Size([2, 6, 7])


#### Visualizing transpose, ( when input in batches)

In [98]:
x = torch.tensor([
    [
        [1, 2], ## seq 1 token 1 
        [3, 4],
        [5, 6],
        [7, 8],
        [9, 10],
        [11, 12] ## seq 1 token 6
    ],
    [
        [13, 14],## seq 2 token 1 
        [15, 16],
        [17, 18],
        [19, 20],
        [21, 22],
        [23, 24] ## seq 2 token 6 
    ]
], dtype=torch.float32)

In [99]:
x.transpose(1,2) ## transposes seqeuence wise

tensor([[[ 1.,  3.,  5.,  7.,  9., 11.],
         [ 2.,  4.,  6.,  8., 10., 12.]],

        [[13., 15., 17., 19., 21., 23.],
         [14., 16., 18., 20., 22., 24.]]])